# Android Random Forest runtime compatibility

This notebook creates a separate Android Random Forest artifact using the existing source-row aligned candidate data and saved split manifest. It does not rebuild connected components or generate a split. Training uses only the manifest's train representatives; validation and test remain their recorded partitions.

The target mapping is provisional: OUTPUT 0 means non-malware and OUTPUT 1 means malware. The source publisher does not document polarity, so this experiment does not independently verify the labels. Results are within-dataset results and do not establish real-world performance.

Run cells in order. The notebook checks that scikit-learn is exactly 1.8.0 and stops before fitting if any alignment, schema, exclusion, feature, or split-isolation check fails. The final cell writes to new paths only and refuses to overwrite them. No cells have been executed as part of notebook creation.

## 1. Runtime and reproducibility

Use the same Python environment as the FastAPI runtime, with scikit-learn 1.8.0. The model is trained here only after the runtime check; the API will not train it. A fixed estimator seed of 20261026 and the saved train/validation/test assignments are used throughout.

The existing metrics file currently records Python 3.14.6 and scikit-learn 1.8.0. The project context also says the earlier Android notebook used scikit-learn 1.9.1. This provenance discrepancy is recorded for review; this notebook records the runtime it actually uses and does not load or overwrite the earlier model.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import platform
import sys
import time

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline

SEED = 20261026
THRESHOLD = 0.5
EXPECTED_SKLEARN = "1.8.0"

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (
    ROOT / "backend" / "data" / "processed" / "android_permission_candidates.csv"
).is_file():
    ROOT = ROOT.parent
if not (ROOT / "backend" / "data" / "processed" / "android_permission_candidates.csv").is_file():
    raise RuntimeError("Start Jupyter from this repository or one of its subdirectories.")

PROCESSED = ROOT / "backend" / "data" / "processed"
SOURCE_REPORTS = ROOT / "backend" / "reports" / "ml" / "android_malware"
CANDIDATE_CSV = PROCESSED / "android_permission_candidates.csv"
CONFIG_PATH = SOURCE_REPORTS / "experiment_config.json"
METRICS_PATH = SOURCE_REPORTS / "metrics.json"
MANIFEST_PATH = SOURCE_REPORTS / "split_manifest.csv"

MODEL_PATH = ROOT / "backend" / "models" / "android" / (
    "android_random_forest_grouped_sklearn_1_8_0.joblib"
)
OUTPUT_DIR = ROOT / "backend" / "reports" / "ml" / "android_malware_sklearn_1_8_0"
OUTPUT_CONFIG = OUTPUT_DIR / "experiment_config.json"
OUTPUT_METRICS = OUTPUT_DIR / "metrics.json"
OUTPUT_REPORT = OUTPUT_DIR / "experiment_report.md"

if sklearn.__version__ != EXPECTED_SKLEARN:
    raise RuntimeError(
        f"Expected scikit-learn {EXPECTED_SKLEARN}; runtime has {sklearn.__version__}. "
        "Stop here and select the FastAPI-compatible environment. Do not install or upgrade packages in this notebook."
    )

required_inputs = [CANDIDATE_CSV, CONFIG_PATH, METRICS_PATH, MANIFEST_PATH]
missing_inputs = [str(p) for p in required_inputs if not p.is_file()]
if missing_inputs:
    raise FileNotFoundError("Required existing inputs are missing: " + ", ".join(missing_inputs))

existing_outputs = [str(p) for p in [MODEL_PATH, OUTPUT_CONFIG, OUTPUT_METRICS, OUTPUT_REPORT] if p.exists()]
if existing_outputs:
    raise FileExistsError("Refusing to overwrite separate output artifacts: " + ", ".join(existing_outputs))

versions = {
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}
print(json.dumps({
    "repository": str(ROOT),
    "versions": versions,
    "seed": SEED,
    "threshold": THRESHOLD,
    "input_paths": [str(p) for p in required_inputs],
    "new_output_paths": [str(p) for p in [MODEL_PATH, OUTPUT_CONFIG, OUTPUT_METRICS, OUTPUT_REPORT]],
}, indent=2))

{
  "repository": "C:\\Projects\\cyber-security-awareness-platform",
  "versions": {
    "python": "3.14.6",
    "python_executable": "c:\\Users\\Sushanth CD\\AppData\\Local\\Programs\\Python\\Python314\\python.exe",
    "numpy": "2.4.4",
    "pandas": "3.0.2",
    "scipy": "1.17.1",
    "scikit_learn": "1.8.0",
    "joblib": "1.5.3"
  },
  "seed": 20261026,
  "threshold": 0.5,
  "input_paths": [
    "C:\\Projects\\cyber-security-awareness-platform\\backend\\data\\processed\\android_permission_candidates.csv",
    "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\experiment_config.json",
    "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\metrics.json",
    "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware\\split_manifest.csv"
  ],
  "new_output_paths": [
    "C:\\Projects\\cyber-security-awareness-platform\\backend\\models\\android\\android_random_forest_grouped_sklear

## 2. Load recorded experiment metadata and inspect schemas

The recorded configuration provides the retained feature list, threshold, seed, split description, and Random Forest parameters. Metrics are read for recorded class counts and a comparison after the new model is evaluated. The existing model files are not loaded or modified.

In [2]:
with CONFIG_PATH.open(encoding="utf-8") as f:
    saved_config = json.load(f)
with METRICS_PATH.open(encoding="utf-8") as f:
    saved_metrics = json.load(f)

assert saved_config["seed"] == SEED
assert float(saved_config["threshold"]) == THRESHOLD
assert saved_metrics["seed"] == SEED
assert float(saved_metrics["threshold"]) == THRESHOLD
assert saved_config["split"]["method"] == saved_metrics["split"]["method"]
assert saved_config["split"]["fold_mapping"] == saved_metrics["split"]["fold_mapping"]

REQUIRED_CANDIDATE_METADATA = [
    "source_row_index", "label_original", "source_dataset",
    "review_required", "review_reason",
]
candidate_header = pd.read_csv(CANDIDATE_CSV, nrows=0).columns.tolist()
assert candidate_header[:5] == REQUIRED_CANDIDATE_METADATA, candidate_header[:5]
candidate_feature_columns = candidate_header[5:]
assert len(candidate_feature_columns) == saved_metrics["feature_preparation"]["input_features"]
assert len(candidate_feature_columns) == 1427

FEATURE_COLUMNS = saved_config["feature_columns"]
assert FEATURE_COLUMNS == saved_metrics["feature_preparation"]["retained_features"]
assert len(FEATURE_COLUMNS) == saved_metrics["feature_preparation"]["retained_feature_count"]
assert len(FEATURE_COLUMNS) == 226
assert len(set(FEATURE_COLUMNS)) == len(FEATURE_COLUMNS)
assert set(FEATURE_COLUMNS).issubset(candidate_feature_columns)

rf_config = saved_config["models"]["random_forest"]
assert rf_config == saved_metrics["models"]["random_forest"]["config"]
required_rf_keys = {
    "n_estimators", "max_features", "min_samples_leaf",
    "class_weight", "random_state", "n_jobs",
}
assert required_rf_keys.issubset(rf_config)
assert rf_config["random_state"] == SEED

print({
    "candidate_csv_columns": len(candidate_header),
    "candidate_metadata": candidate_header[:5],
    "candidate_input_feature_count": len(candidate_feature_columns),
    "retained_feature_count": len(FEATURE_COLUMNS),
    "recorded_random_forest_config": rf_config,
    "saved_split_method": saved_config["split"]["method"],
    "saved_fold_mapping": saved_config["split"]["fold_mapping"],
    "provisional_target_mapping": saved_metrics["target_mapping"],
    "recorded_runtime_versions": saved_metrics["versions"],
})

{'candidate_csv_columns': 1432, 'candidate_metadata': ['source_row_index', 'label_original', 'source_dataset', 'review_required', 'review_reason'], 'candidate_input_feature_count': 1427, 'retained_feature_count': 226, 'recorded_random_forest_config': {'n_estimators': 200, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample', 'random_state': 20261026, 'n_jobs': -1}, 'saved_split_method': 'StratifiedGroupKFold with 20 folds; whole package/feature connected components', 'saved_fold_mapping': '14 train folds, 3 validation folds, 3 test folds', 'provisional_target_mapping': {'0': 'non-malware (provisional)', '1': 'malware (provisional)', 'status': 'provisional; externally corroborated but not publisher-confirmed'}, 'recorded_runtime_versions': {'python': '3.14.6', 'numpy': '2.4.4', 'pandas': '3.0.2', 'scipy': '1.17.1', 'scikit_learn': '1.8.0', 'joblib': '1.5.3'}}


## 3. Read and validate the existing split manifest

The manifest is the sole source of partition assignment. This cell checks its required fields, unique source-row keys, known partitions, labels, representative and exclusion flags, and agreement with the previously recorded row and class counts. A representative is the saved modeling row for an exact complete record; non-representatives remain in the manifest and are not silently removed.

In [3]:
MANIFEST_COLUMNS = [
    "source_row_index", "source_dataset", "label_original",
    "package_id_for_grouping_only", "feature_vector_sha256", "component_id",
    "partition", "review_required", "review_reason", "exclusion_reason",
    "exact_record_duplicate", "model_observation_representative",
]
manifest = pd.read_csv(
    MANIFEST_PATH, usecols=MANIFEST_COLUMNS, dtype=str, keep_default_na=False
)
manifest["source_row_index"] = pd.to_numeric(
    manifest["source_row_index"], errors="raise"
).astype("int64")
if manifest["source_row_index"].duplicated().any():
    raise ValueError("Split manifest contains duplicate source_row_index values.")
if set(manifest["label_original"]) != {"0", "1"}:
    raise ValueError("Manifest labels must be exactly the original binary labels 0 and 1.")
EXPECTED_PARTITIONS = {"train", "validation", "test", "quarantined"}
if not set(manifest["partition"]).issubset(EXPECTED_PARTITIONS):
    raise ValueError(f"Unexpected manifest partitions: {set(manifest['partition']) - EXPECTED_PARTITIONS}")
if not set(manifest["partition"]) >= {"train", "validation", "test", "quarantined"}:
    raise ValueError("One or more expected saved partitions are absent.")

def parse_manifest_bool(values, field_name):
    normalized = values.astype(str).str.strip().str.casefold()
    if not normalized.isin({"true", "false"}).all():
        raise ValueError(f"Unexpected boolean representation in manifest field {field_name}.")
    return normalized.eq("true")

for bool_field in ["review_required", "exact_record_duplicate", "model_observation_representative"]:
    manifest[bool_field] = parse_manifest_bool(manifest[bool_field], bool_field)

if manifest.loc[manifest["partition"].eq("quarantined"), "model_observation_representative"].any():
    raise ValueError("A quarantined record is marked as a model observation representative.")
if manifest.loc[manifest["model_observation_representative"], "partition"].eq("quarantined").any():
    raise ValueError("A representative is assigned to the quarantined partition.")
if manifest.loc[manifest["model_observation_representative"], "exclusion_reason"].ne("").any():
    raise ValueError("An excluded record is marked as a model observation representative.")
if manifest.loc[manifest["model_observation_representative"], "exact_record_duplicate"].any():
    raise ValueError("An exact duplicate is marked as the representative.")
if manifest.loc[
    manifest["partition"].isin({"train", "validation", "test"}) &
    manifest["exclusion_reason"].ne(""),
    "source_row_index"
].size:
    raise ValueError("A record with an exclusion reason appears in a primary partition.")

recorded_counts = saved_metrics["split"]["record_counts_and_classes"]
observed_partition_counts = {}
for partition in ["train", "validation", "test"]:
    part = manifest.loc[manifest["partition"].eq(partition)]
    reps = part.loc[part["model_observation_representative"]]
    observed_partition_counts[partition] = {
        "rows": int(len(part)),
        "label_counts": {str(k): int(v) for k, v in part["label_original"].value_counts().sort_index().items()},
        "representatives": int(len(reps)),
        "representative_label_counts": {
            str(k): int(v) for k, v in reps["label_original"].value_counts().sort_index().items()
        },
    }
    expected = recorded_counts[partition]
    assert len(part) == expected["rows"], (partition, len(part), expected["rows"])
    assert len(reps) == expected["unique_complete_records_for_model"]
    assert observed_partition_counts[partition]["representative_label_counts"] == {
        str(k): int(v) for k, v in expected["unique_record_label_counts"].items()
    }

quarantine_count = int(manifest["partition"].eq("quarantined").sum())
assert quarantine_count == saved_metrics["duplicate_conflict_policy"]["quarantined_source_rows"]
print({
    "manifest_rows": len(manifest),
    "quarantined_rows": quarantine_count,
    "quarantined_components": int(
        manifest.loc[manifest["partition"].eq("quarantined"), "component_id"].nunique()
    ),
    "partition_counts": observed_partition_counts,
    "review_required_rows": int(manifest["review_required"].sum()),
    "note": "review_required includes the dataset-wide provisional-label review flag; model exclusions follow the saved manifest.",
})

{'manifest_rows': 121189, 'quarantined_rows': 99234, 'quarantined_components': 999, 'partition_counts': {'train': {'rows': 15367, 'label_counts': {'0': 13987, '1': 1380}, 'representatives': 7321, 'representative_label_counts': {'0': 6039, '1': 1282}}, 'validation': {'rows': 3294, 'label_counts': {'0': 2997, '1': 297}, 'representatives': 1577, 'representative_label_counts': {'0': 1301, '1': 276}}, 'test': {'rows': 3294, 'label_counts': {'0': 2998, '1': 296}, 'representatives': 1609, 'representative_label_counts': {'0': 1327, '1': 282}}}, 'review_required_rows': 121189, 'note': 'review_required includes the dataset-wide provisional-label review flag; model exclusions follow the saved manifest.'}


## 4. Verify source-row alignment and feature integrity before fitting

The candidate CSV omits package IDs and exclusion decisions, so those are taken from the saved manifest, keyed by source_row_index. This streaming pass verifies every candidate record against that manifest and recomputes the saved full-vector hash using the same packing convention as the original notebook. It also checks binary values and accumulates only the recorded representative rows using the saved 226-feature list.

This is a data-integrity scan only: it does not recreate groups, components, or partitions. It may take several minutes. No model is fitted in this cell.

In [4]:
CHUNK_SIZE = 512
manifest_by_row = manifest.set_index("source_row_index", drop=False)
if len(manifest_by_row) != len(manifest):
    raise ValueError("Manifest row index is not one-to-one.")

X_chunks = {p: [] for p in ["train", "validation", "test"]}
y_chunks = {p: [] for p in ["train", "validation", "test"]}
id_chunks = {p: [] for p in ["train", "validation", "test"]}
seen_source_ids = set()
rows_scanned = 0
source_names = set()
started = time.perf_counter()

for chunk_number, chunk in enumerate(pd.read_csv(
    CANDIDATE_CSV, chunksize=CHUNK_SIZE, dtype=str, keep_default_na=False
), start=1):
    ids = pd.to_numeric(chunk["source_row_index"], errors="raise").astype("int64").to_numpy()
    if len(set(ids.tolist())) != len(ids) or any(int(i) in seen_source_ids for i in ids):
        raise ValueError(f"Candidate source_row_index is duplicated in chunk {chunk_number}.")
    missing_ids = [int(i) for i in ids if int(i) not in manifest_by_row.index]
    if missing_ids:
        raise ValueError(f"Candidate rows are absent from the manifest: {missing_ids[:10]}")
    seen_source_ids.update(int(i) for i in ids)

    expected = manifest_by_row.loc[ids].reset_index(drop=True)
    if not np.array_equal(expected["source_row_index"].to_numpy(dtype=np.int64), ids):
        raise ValueError("Source-row keyed manifest alignment failed.")

    labels = chunk["label_original"].astype(str).to_numpy()
    sources = chunk["source_dataset"].astype(str).to_numpy()
    review_reasons = chunk["review_reason"].astype(str).to_numpy()
    candidate_review = chunk["review_required"].astype(str).str.strip().str.casefold().to_numpy()
    manifest_review = expected["review_required"].astype(str).str.strip().str.casefold().to_numpy()
    if not np.array_equal(labels, expected["label_original"].to_numpy(dtype=str)):
        raise ValueError(f"Original labels do not align for chunk {chunk_number}.")
    if not np.array_equal(sources, expected["source_dataset"].to_numpy(dtype=str)):
        raise ValueError(f"Source provenance does not align for chunk {chunk_number}.")
    if not np.array_equal(candidate_review, manifest_review):
        raise ValueError(f"Review flags do not align for chunk {chunk_number}.")
    if not np.array_equal(review_reasons, expected["review_reason"].to_numpy(dtype=str)):
        raise ValueError(f"Review reasons do not align for chunk {chunk_number}.")
    if not set(labels).issubset({"0", "1"}):
        raise ValueError(f"Unexpected original label in chunk {chunk_number}.")
    if not candidate_review.size == len(ids) or not set(candidate_review).issubset({"true", "false"}):
        raise ValueError(f"Unexpected review_required values in chunk {chunk_number}.")
    source_names.update(sources.tolist())

    text_features = chunk[candidate_feature_columns].to_numpy(dtype=str)
    if text_features.size and not np.isin(text_features, ["0", "1"]).all():
        bad = sorted(set(text_features.reshape(-1)) - {"0", "1"})
        raise ValueError(f"Missing or nonbinary feature values in chunk {chunk_number}: {bad[:10]}")
    binary = text_features.astype(np.uint8, copy=False)
    packed_rows = np.packbits(binary, axis=1, bitorder="little")
    observed_hashes = [
        hashlib.sha256(packed.tobytes()).hexdigest() for packed in packed_rows
    ]
    if not np.array_equal(
        np.asarray(observed_hashes, dtype=object),
        expected["feature_vector_sha256"].to_numpy(dtype=object),
    ):
        raise ValueError(f"Feature-vector hashes do not align with the manifest in chunk {chunk_number}.")

    row_partitions = expected["partition"].to_numpy(dtype=str)
    representative = expected["model_observation_representative"].to_numpy(dtype=bool)
    for partition in ["train", "validation", "test"]:
        keep = (row_partitions == partition) & representative
        if keep.any():
            selected = chunk.loc[keep, FEATURE_COLUMNS].to_numpy(dtype=np.uint8)
            X_chunks[partition].append(selected)
            y_chunks[partition].append(labels[keep].astype(np.int8))
            id_chunks[partition].append(ids[keep])

    rows_scanned += len(chunk)
    if chunk_number % 40 == 0:
        print(f"Verified {rows_scanned:,} candidate rows")

if rows_scanned != len(manifest) or seen_source_ids != set(manifest["source_row_index"].tolist()):
    raise ValueError(
        f"Candidate/manifest coverage mismatch: scanned={rows_scanned:,}, "
        f"manifest={len(manifest):,}, missing={len(set(manifest['source_row_index']) - seen_source_ids):,}"
    )
if source_names != {"android_benign_malware_dataset2"}:
    raise ValueError(f"Unexpected source_dataset values: {source_names}")

X = {p: np.vstack(X_chunks[p]) for p in ["train", "validation", "test"]}
y = {p: np.concatenate(y_chunks[p]) for p in ["train", "validation", "test"]}
source_ids = {p: np.concatenate(id_chunks[p]) for p in ["train", "validation", "test"]}
for partition in ["train", "validation", "test"]:
    expected_count = recorded_counts[partition]["unique_complete_records_for_model"]
    if X[partition].shape != (expected_count, len(FEATURE_COLUMNS)):
        raise ValueError(
            f"{partition} matrix shape {X[partition].shape} differs from "
            f"expected {(expected_count, len(FEATURE_COLUMNS))}."
        )
    if len(set(source_ids[partition].tolist())) != len(source_ids[partition]):
        raise ValueError(f"Duplicate source rows entered {partition}.")
    if set(np.unique(y[partition]).tolist()) != {0, 1}:
        raise ValueError(f"{partition} does not contain both original labels.")

print({
    "verified_source_rows": rows_scanned,
    "source_dataset_values": sorted(source_names),
    "primary_matrix_shapes": {p: list(X[p].shape) for p in X},
    "primary_class_counts": {
        p: {str(k): int(v) for k, v in Counter(y[p].tolist()).items()} for p in y
    },
    "scan_seconds": round(time.perf_counter() - started, 2),
})

Verified 20,480 candidate rows
Verified 40,960 candidate rows
Verified 61,440 candidate rows
Verified 81,920 candidate rows
Verified 102,400 candidate rows
{'verified_source_rows': 121189, 'source_dataset_values': ['android_benign_malware_dataset2'], 'primary_matrix_shapes': {'train': [7321, 226], 'validation': [1577, 226], 'test': [1609, 226]}, 'primary_class_counts': {'train': {'0': 6039, '1': 1282}, 'validation': {'0': 1301, '1': 276}, 'test': {'0': 1327, '1': 282}}, 'scan_seconds': 130.19}


## 5. Verify grouping, quarantine, and exclusion isolation

No connected-component calculation is performed. The notebook validates the saved component assignments and checks whether component IDs, non-empty package IDs, and exact feature-vector hashes span multiple partitions. Quarantined records must remain outside the primary partitions. The package identifier is used only for this audit and is never included in the model features.

In [5]:
PRIMARY_PARTITIONS = {"train", "validation", "test"}
primary_rows = manifest.loc[manifest["partition"].isin(PRIMARY_PARTITIONS)].copy()

def assert_group_isolation(frame, column, ignore_empty=False):
    values = frame[column].astype(str)
    if ignore_empty:
        frame = frame.loc[values.str.strip().ne("")]
    cross_partition = (
        frame.groupby(column, dropna=False)["partition"].nunique().loc[lambda s: s > 1]
    )
    if not cross_partition.empty:
        examples = cross_partition.index[:10].tolist()
        raise ValueError(f"{column} crosses partitions; example keys: {examples}")
    return int(frame[column].nunique())

component_groups = assert_group_isolation(manifest, "component_id")
package_groups = assert_group_isolation(primary_rows, "package_id_for_grouping_only", ignore_empty=True)
feature_groups = assert_group_isolation(manifest, "feature_vector_sha256")
assert_group_isolation(manifest, "component_id")
if manifest.groupby("component_id")["partition"].nunique().gt(1).any():
    raise ValueError("A saved component connects a quarantined row to a primary partition.")
if manifest.loc[manifest["partition"].eq("quarantined"), "model_observation_representative"].any():
    raise ValueError("A quarantined record entered the modeling representative set.")
if primary_rows.loc[primary_rows["model_observation_representative"], "exclusion_reason"].ne("").any():
    raise ValueError("An excluded source record is included in model observations.")

isolation_checks = {
    "component_ids_partition_isolated": True,
    "package_ids_partition_isolated": True,
    "feature_vector_hashes_partition_isolated": True,
    "quarantine_components_separate_from_primary": True,
    "no_quarantined_model_representatives": True,
    "no_excluded_model_representatives": True,
    "component_group_count": component_groups,
    "primary_nonempty_package_group_count": package_groups,
    "feature_vector_group_count": feature_groups,
    "records_without_package_id_for_grouping": int(
        primary_rows["package_id_for_grouping_only"].astype(str).str.strip().eq("").sum()
    ),
    "quarantined_source_rows": int(manifest["partition"].eq("quarantined").sum()),
    "exact_duplicate_nonrepresentatives_in_primary": int(
        (primary_rows["exact_record_duplicate"] & ~primary_rows["model_observation_representative"]).sum()
    ),
}
assert saved_config["split"]["component_isolation_verified"] is True
assert saved_config["split"]["package_isolation_verified"] is True
assert saved_config["split"]["feature_vector_isolation_verified"] is True
assert saved_config["split"]["quarantine_isolation_verified"] is True
print(json.dumps(isolation_checks, indent=2))

{
  "component_ids_partition_isolated": true,
  "package_ids_partition_isolated": true,
  "feature_vector_hashes_partition_isolated": true,
  "quarantine_components_separate_from_primary": true,
  "no_quarantined_model_representatives": true,
  "no_excluded_model_representatives": true,
  "component_group_count": 7256,
  "primary_nonempty_package_group_count": 10211,
  "feature_vector_group_count": 12222,
  "records_without_package_id_for_grouping": 0,
  "quarantined_source_rows": 99234,
  "exact_duplicate_nonrepresentatives_in_primary": 11448
}


## 6. Training gate and recorded Random Forest

Training is blocked until all alignment and isolation checks above have completed successfully. No split is generated here. The Random Forest uses the recorded parameters; max_samples=None is explicit to match the original notebook's estimator construction. The fixed 0.5 threshold is for reporting only and is not tuned on validation or test.

In [6]:
assert sklearn.__version__ == EXPECTED_SKLEARN == "1.8.0"
assert rows_scanned == len(manifest)
assert set(X) == {"train", "validation", "test"}
assert all(X[p].shape[1] == len(FEATURE_COLUMNS) for p in X)
assert all(len(X[p]) == len(y[p]) == len(source_ids[p]) for p in X)
assert isolation_checks["component_ids_partition_isolated"]
assert isolation_checks["package_ids_partition_isolated"]
assert isolation_checks["feature_vector_hashes_partition_isolated"]
assert isolation_checks["quarantine_components_separate_from_primary"]
assert all(p.exists() for p in [CANDIDATE_CSV, CONFIG_PATH, METRICS_PATH, MANIFEST_PATH])
assert not any(p.exists() for p in [MODEL_PATH, OUTPUT_CONFIG, OUTPUT_METRICS, OUTPUT_REPORT])

# Existing recorded settings, plus max_samples=None from the original notebook.
rf_fit_config = {
    **rf_config,
    "max_samples": None,
}
print("Training gate passed. Training rows:", len(y["train"]))
print("Parameters:", rf_fit_config)

Training gate passed. Training rows: 7321
Parameters: {'n_estimators': 200, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample', 'random_state': 20261026, 'n_jobs': -1, 'max_samples': None}


## 7. Fit on train only; evaluate saved validation and test partitions

The model is fitted only on the existing train representatives. Validation and test are prediction-only partitions. This notebook does not select hyperparameters or change the decision threshold. Metrics are reported separately at the recorded threshold.

In [7]:
model = Pipeline([
    ("classifier", RandomForestClassifier(**rf_fit_config))
])

fit_started = time.perf_counter()
model.fit(X["train"], y["train"])
fit_seconds = time.perf_counter() - fit_started

def evaluate_partition(estimator, features, labels, threshold=THRESHOLD):
    classifier = estimator.named_steps["classifier"]
    classes = list(classifier.classes_)
    scores = estimator.predict_proba(features)[:, classes.index(1)]
    predictions = (scores >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    result = {
        "records": int(len(labels)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else None,
        "accuracy": float(accuracy_score(labels, predictions)),
        "roc_auc": (
            float(roc_auc_score(labels, scores))
            if len(np.unique(labels)) == 2 else None
        ),
        "threshold": float(threshold),
    }
    return result, scores, predictions

partition_results = {}
partition_outputs = {}
for partition in ["validation", "test"]:
    metrics, scores, predictions = evaluate_partition(model, X[partition], y[partition])
    partition_results[partition] = metrics
    partition_outputs[partition] = {
        "scores": scores,
        "predictions": predictions,
    }

print(json.dumps({
    "fit_seconds": round(fit_seconds, 2),
    "training_records": int(len(y["train"])),
    "training_label_counts": {str(k): int(v) for k, v in Counter(y["train"].tolist()).items()},
    "validation_metrics": partition_results["validation"],
    "test_metrics": partition_results["test"],
}, indent=2))

{
  "fit_seconds": 0.8,
  "training_records": 7321,
  "training_label_counts": {
    "0": 6039,
    "1": 1282
  },
  "validation_metrics": {
    "records": 1577,
    "tn": 1283,
    "fp": 18,
    "fn": 23,
    "tp": 253,
    "precision": 0.933579335793358,
    "recall": 0.9166666666666666,
    "f1": 0.9250457038391224,
    "specificity": 0.9861644888547272,
    "accuracy": 0.974001268230818,
    "roc_auc": 0.9865460236830086,
    "threshold": 0.5
  },
  "test_metrics": {
    "records": 1609,
    "tn": 1312,
    "fp": 15,
    "fn": 18,
    "tp": 264,
    "precision": 0.946236559139785,
    "recall": 0.9361702127659575,
    "f1": 0.9411764705882353,
    "specificity": 0.9886963074604371,
    "accuracy": 0.9794903666873834,
    "roc_auc": 0.9895647944758881,
    "threshold": 0.5
  }
}


## 8. Compare with the recorded experiment on the same partitions

This comparison uses the existing metrics file's saved validation and test results. It does not change the model, split, or parameters. Treat any difference as an observed reproduction difference; the source metrics currently list scikit-learn 1.8.0, while project context has described the previous notebook runtime as 1.9.1. The earlier model artifact itself is not loaded.

In [8]:
previous_rf_metrics = saved_metrics["models"]["random_forest"]
comparison = {}
for partition in ["validation", "test"]:
    comparison[partition] = {
        "recorded_metrics_version": saved_metrics["versions"]["scikit_learn"],
        "recorded": previous_rf_metrics[f"{partition}_metrics"],
        "retrained_under_runtime_1_8_0": partition_results[partition],
    }
print(json.dumps(comparison, indent=2))

{
  "validation": {
    "recorded_metrics_version": "1.8.0",
    "recorded": {
      "records": 1577,
      "tn": 1281,
      "fp": 20,
      "fn": 25,
      "tp": 251,
      "precision": 0.9261992619926199,
      "recall": 0.9094202898550725,
      "f1": 0.9177330895795247,
      "specificity": 0.9846272098385856,
      "accuracy": 0.9714648065948003,
      "roc_auc": 0.9869832570263677
    },
    "retrained_under_runtime_1_8_0": {
      "records": 1577,
      "tn": 1283,
      "fp": 18,
      "fn": 23,
      "tp": 253,
      "precision": 0.933579335793358,
      "recall": 0.9166666666666666,
      "f1": 0.9250457038391224,
      "specificity": 0.9861644888547272,
      "accuracy": 0.974001268230818,
      "roc_auc": 0.9865460236830086,
      "threshold": 0.5
    }
  },
  "test": {
    "recorded_metrics_version": "1.8.0",
    "recorded": {
      "records": 1609,
      "tn": 1308,
      "fp": 19,
      "fn": 18,
      "tp": 264,
      "precision": 0.9328621908127208,
      "recall": 0.

## 9. Save separate artifacts

The model and reports are written only to the new output paths shown below. Existing Random Forest, Logistic Regression, split manifest, metrics, and reports remain untouched. The configuration records the exact runtime, parameters, selected features, and saved partition counts. The report labels the results as provisional, within-dataset evaluation.

In [9]:
from datetime import datetime, timezone

# Recheck immediately before writing; never replace an existing artifact.
existing_outputs = [str(p) for p in [MODEL_PATH, OUTPUT_CONFIG, OUTPUT_METRICS, OUTPUT_REPORT] if p.exists()]
if existing_outputs:
    raise FileExistsError("Refusing to overwrite separate output artifacts: " + ", ".join(existing_outputs))

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

new_config = {
    "experiment_id": "android-random-forest-grouped-sklearn-1.8.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "threshold": THRESHOLD,
    "target_mapping_note": saved_config["target_mapping_note"],
    "label_policy": "Preserve original OUTPUT labels; mapping remains provisional and is not independently verified.",
    "input_candidate_csv": str(CANDIDATE_CSV),
    "input_manifest": str(MANIFEST_PATH),
    "input_source_config": str(CONFIG_PATH),
    "input_source_metrics": str(METRICS_PATH),
    "versions": versions,
    "feature_columns": FEATURE_COLUMNS,
    "feature_count": len(FEATURE_COLUMNS),
    "excluded_model_features": ["package_id_for_grouping_only", "category_audit_only"],
    "split": {
        "reused": True,
        "method": saved_config["split"]["method"],
        "fold_mapping": saved_config["split"]["fold_mapping"],
        "partition_counts": observed_partition_counts,
        "isolation_checks": isolation_checks,
    },
    "random_forest": rf_fit_config,
    "fit_partition": "train",
    "evaluation_partitions": ["validation", "test"],
    "test_used_for_tuning": False,
    "provenance_note": (
        "Existing metrics record scikit-learn "
        + saved_metrics["versions"]["scikit_learn"]
        + "; project notes also describe the earlier notebook runtime as 1.9.1. "
          "This artifact records the runtime used for this fit."
    ),
    "artifacts": {
        "model": str(MODEL_PATH),
        "config": str(OUTPUT_CONFIG),
        "metrics": str(OUTPUT_METRICS),
        "report": str(OUTPUT_REPORT),
    },
}

new_metrics = {
    "experiment_id": new_config["experiment_id"],
    "versions": versions,
    "seed": SEED,
    "threshold": THRESHOLD,
    "target_mapping": saved_metrics["target_mapping"],
    "interpretation": "Provisional within-dataset evaluation; not independent real-world validation.",
    "records": {
        "training": {
            "count": int(len(y["train"])),
            "label_counts": {str(k): int(v) for k, v in Counter(y["train"].tolist()).items()},
        },
        "validation": {
            "count": int(len(y["validation"])),
            "label_counts": {str(k): int(v) for k, v in Counter(y["validation"].tolist()).items()},
            "metrics": partition_results["validation"],
        },
        "test": {
            "count": int(len(y["test"])),
            "label_counts": {str(k): int(v) for k, v in Counter(y["test"].tolist()).items()},
            "metrics": partition_results["test"],
        },
        "quarantined_source_rows": int(quarantine_count),
    },
    "fit_seconds": float(fit_seconds),
    "split_isolation_checks": isolation_checks,
    "comparison_to_recorded_metrics": comparison,
    "artifacts": new_config["artifacts"],
}

# Use exclusive creation for reports; the model destination was confirmed absent.
joblib.dump(model, MODEL_PATH)
with OUTPUT_CONFIG.open("x", encoding="utf-8") as f:
    json.dump(new_config, f, indent=2)
    f.write("\n")
with OUTPUT_METRICS.open("x", encoding="utf-8") as f:
    json.dump(new_metrics, f, indent=2)
    f.write("\n")

report_lines = [
    "# Android Random Forest — scikit-learn 1.8.0 compatibility run",
    "",
    "## Scope and labels",
    "This is a within-dataset reproduction using saved train, validation, and test assignments. It is not independent real-world validation.",
    saved_config["target_mapping_note"],
    "Original OUTPUT labels were preserved. Their polarity is provisional and not independently verified by the dataset publisher.",
    "",
    "## Runtime and model",
    f"- Python: {versions['python']}",
    f"- scikit-learn: {versions['scikit_learn']}",
    f"- NumPy / pandas / SciPy / joblib: {versions['numpy']} / {versions['pandas']} / {versions['scipy']} / {versions['joblib']}",
    f"- Seed: {SEED}; threshold: {THRESHOLD}.",
    f"- Random Forest parameters: {json.dumps(rf_fit_config, sort_keys=True)}",
    f"- Retained features: {len(FEATURE_COLUMNS)}. Package and category fields are excluded from model inputs.",
    "",
    "## Data and split verification",
    f"- Candidate rows aligned to manifest: {rows_scanned:,}.",
    f"- Train / validation / test model representatives: {len(y['train']):,} / {len(y['validation']):,} / {len(y['test']):,}.",
    f"- Quarantined source rows retained outside modeling: {quarantine_count:,}.",
    "- Grouping was not recomputed; the saved split manifest was reused and its component, package, feature-vector, and quarantine isolation were checked.",
    "",
    "## Results",
    f"- Validation: {json.dumps(partition_results['validation'], sort_keys=True)}",
    f"- Test: {json.dumps(partition_results['test'], sort_keys=True)}",
    "",
    "These are dataset-source results under provisional labels. They do not establish real-world Android malware detection accuracy.",
    "",
    "## Reproducibility note",
    "The original metrics record scikit-learn "
    + str(saved_metrics["versions"]["scikit_learn"])
    + ", while project context also describes the prior notebook runtime as 1.9.1. This report records the runtime used for this fit; inspect source artifact provenance before attributing differences to library compatibility.",
]
with OUTPUT_REPORT.open("x", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

print(json.dumps(new_config["artifacts"], indent=2))

{
  "model": "C:\\Projects\\cyber-security-awareness-platform\\backend\\models\\android\\android_random_forest_grouped_sklearn_1_8_0.joblib",
  "config": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware_sklearn_1_8_0\\experiment_config.json",
  "metrics": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware_sklearn_1_8_0\\metrics.json",
  "report": "C:\\Projects\\cyber-security-awareness-platform\\backend\\reports\\ml\\android_malware_sklearn_1_8_0\\experiment_report.md"
}


## Verification requirements before using this model in FastAPI

After saving, verify the recorded runtime is exactly scikit-learn 1.8.0 and that the new pipeline loads in the FastAPI runtime with that same environment. Then perform an API-level smoke test that uses the model without training. Keep the previous Android models and metrics for comparison. No detector or FastAPI files are modified by this notebook.

Do not interpret this dataset-source result as real-world accuracy. OUTPUT label polarity is provisional, and source collection bias and duplicate/conflict quarantine remain limitations.